# Laboratorio 8 — Preparación, descarga y exploración con DuckDB

**Objetivo:** dejar un ambiente reproducible, descargar los trip records Parquet
de taxis amarillos y verdes de 2026 publicados por la TLC, verificar integridad
y explorar los archivos directamente con DuckDB (sin materializar todavía una
base persistente).


## Configuración y rutas

Se localiza la raíz del proyecto buscando `docker-compose.yml` hacia arriba desde
el directorio de trabajo actual. Así el notebook funciona dentro de Docker
(`/workspace`) y también en ejecución local.


In [1]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow as pa


def find_project_root(start: Path | None = None) -> Path:
    """Recorre cwd y padres hasta encontrar docker-compose.yml.

    Si no está montado, acepta un directorio con data/, notebooks/ y scripts/.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "docker-compose.yml").is_file():
            return candidate
        markers = (
            (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
            and (candidate / "scripts" / "download_data.py").is_file()
        )
        if markers:
            return candidate
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto; abre el notebook desde el lab."
    )


PROJECT_ROOT = find_project_root()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
SQL_DIR = PROJECT_ROOT / "sql"
DOCS_DIR = PROJECT_ROOT / "docs"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

for path in (DATA_RAW, DATA_PROCESSED, SQL_DIR, DOCS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT   =", PROJECT_ROOT)
print("DATA_RAW       =", DATA_RAW)
print("DATA_PROCESSED =", DATA_PROCESSED)
print("SQL_DIR        =", SQL_DIR)
print("DOCS_DIR       =", DOCS_DIR)


PROJECT_ROOT   = /workspace
DATA_RAW       = /workspace/data/raw
DATA_PROCESSED = /workspace/data/processed
SQL_DIR        = /workspace/sql
DOCS_DIR       = /workspace/docs


## Ambiente reproducible

Servicios definidos en `docker-compose.yml`:

| Servicio | Función | Puerto local |
|---|---|---|
| `analysis` | JupyterLab + Python/DuckDB para el análisis | http://localhost:8888 |
| `metabase` | BI / tableros (configuración posterior) | http://localhost:3000 |

Dentro del contenedor de análisis se montan en `/workspace`:

- `data/`
- `notebooks/`
- `scripts/`
- `sql/`
- `docs/`

Un ambiente reproducible importa porque fija versiones, evita diferencias entre
máquinas y permite volver a ejecutar el análisis sin depender de paquetes
instalados “a mano” en el host.


In [2]:
print("Python :", sys.version.split()[0])
print("DuckDB :", duckdb.__version__)
print("Pandas :", pd.__version__)
print("PyArrow:", pa.__version__)


Python : 3.11.17
DuckDB : 1.1.3
Pandas : 2.2.3
PyArrow: 18.1.0


## Estructura del proyecto

- `data/raw/`: Parquet originales descargados de la TLC.
- `data/processed/`: base DuckDB materializada u otros artefactos generados.
- `notebooks/`: análisis reproducible y documentación ejecutable.
- `scripts/`: automatización reutilizable (descarga).
- `sql/`: consultas SQL importantes organizadas por propósito.
- `docs/`: resultados pequeños, evidencia del tablero o notas auxiliares.


## Inventario de datos 2026

La descarga se hace con `scripts/download_data.py` (yellow y green, año 2026).
Aquí solo se resume el estado local y se verifica completitud respecto a lo
publicado por la TLC.


In [3]:
import importlib.util
import sys

spec = importlib.util.spec_from_file_location(
    "download_data", SCRIPTS_DIR / "download_data.py"
)
download_data = importlib.util.module_from_spec(spec)
assert spec.loader is not None
# Necesario para que @dataclass resuelva anotaciones al cargar vía importlib
sys.modules[spec.name] = download_data
spec.loader.exec_module(download_data)

YEAR = 2026
TAXI_TYPES = ("yellow", "green")

# Resumen de verificación sin volver a descargar
verify_summary = download_data.verify_completeness(
    PROJECT_ROOT, TAXI_TYPES, YEAR
)
is_complete = download_data.print_verification(verify_summary)



=== Verificación de completitud ===
Completo = todos los meses publicados por la TLC están locales y con tamaño > 0 (no se exigen 12 meses).
Publicados:              16
Locales válidos:         16
Ausentes:                0
Locales no publicados:   0
  yellow: 8/8 publicados | meses locales=[1, 2, 3, 4, 5, 6, 7, 8]
  green: 8/8 publicados | meses locales=[1, 2, 3, 4, 5, 6, 7, 8]

Resultado: COMPLETO


In [4]:
rows = []
for taxi_type in TAXI_TYPES:
    year_dir = DATA_RAW / taxi_type / str(YEAR)
    if not year_dir.is_dir():
        continue
    for path in sorted(year_dir.glob("*.parquet")):
        match = re.search(r"(\d{4})-(\d{2})", path.name)
        if not match:
            continue
        year = int(match.group(1))
        month = int(match.group(2))
        size_mb = path.stat().st_size / (1024 * 1024)
        rows.append(
            {
                "taxi_type": taxi_type,
                "year": year,
                "month": month,
                "filename": path.name,
                "size_mb": round(size_mb, 2),
            }
        )

inventory_df = pd.DataFrame(rows).sort_values(["taxi_type", "year", "month"])
display(inventory_df)

n_yellow = int((inventory_df["taxi_type"] == "yellow").sum()) if not inventory_df.empty else 0
n_green = int((inventory_df["taxi_type"] == "green").sum()) if not inventory_df.empty else 0
total_mb = float(inventory_df["size_mb"].sum()) if not inventory_df.empty else 0.0

print(f"Archivos yellow: {n_yellow}")
print(f"Archivos green : {n_green}")
print(f"Tamaño total   : {total_mb:.2f} MB")
print(
    "Meses yellow   :",
    sorted(inventory_df.loc[inventory_df.taxi_type == "yellow", "month"].tolist()),
)
print(
    "Meses green    :",
    sorted(inventory_df.loc[inventory_df.taxi_type == "green", "month"].tolist()),
)
print("Verificación   :", "COMPLETO" if is_complete else "INCOMPLETO")


,taxi_type,year,month,filename,size_mb
8,green,2026,1,green_tripdata_2026-01.parquet,0.95
9,green,2026,2,green_tripdata_2026-02.parquet,0.88
10,green,2026,3,green_tripdata_2026-03.parquet,1.03
11,green,2026,4,green_tripdata_2026-04.parquet,1.03
12,green,2026,5,green_tripdata_2026-05.parquet,1.05
13,green,2026,6,green_tripdata_2026-06.parquet,1.03
14,green,2026,7,green_tripdata_2026-07.parquet,0.97
15,green,2026,8,green_tripdata_2026-08.parquet,0.96
0,yellow,2026,1,yellow_tripdata_2026-01.parquet,61.19
1,yellow,2026,2,yellow_tripdata_2026-02.parquet,55.96


Archivos yellow: 8
Archivos green : 8
Tamaño total   : 495.66 MB
Meses yellow   : [1, 2, 3, 4, 5, 6, 7, 8]
Meses green    : [1, 2, 3, 4, 5, 6, 7, 8]
Verificación   : COMPLETO


## Conexión DuckDB

Conexión **en memoria**. Todavía no se materializa una tabla persistente: se
consulta Parquet con `read_parquet` y vistas temporales normalizadas.


In [5]:
con = duckdb.connect()  # in-memory

yellow_glob = str(DATA_RAW / "yellow" / str(YEAR) / "*.parquet")
green_glob = str(DATA_RAW / "green" / str(YEAR) / "*.parquet")

con.execute(f"""
CREATE OR REPLACE TEMP VIEW yellow_trips AS
SELECT
    'yellow' AS taxi_type,
    tpep_pickup_datetime AS pickup_datetime,
    tpep_dropoff_datetime AS dropoff_datetime,
    passenger_count,
    trip_distance,
    PULocationID AS pu_location_id,
    DOLocationID AS do_location_id,
    payment_type,
    fare_amount,
    tip_amount,
    tolls_amount,
    total_amount,
    filename AS source_file,
    CAST(regexp_extract(filename, '(\d{{4}})-(\d{{2}})', 1) AS INTEGER) AS source_year,
    CAST(regexp_extract(filename, '(\d{{4}})-(\d{{2}})', 2) AS INTEGER) AS source_month
FROM read_parquet('{yellow_glob}', filename = true, union_by_name = true);
""")

con.execute(f"""
CREATE OR REPLACE TEMP VIEW green_trips AS
SELECT
    'green' AS taxi_type,
    lpep_pickup_datetime AS pickup_datetime,
    lpep_dropoff_datetime AS dropoff_datetime,
    passenger_count,
    trip_distance,
    PULocationID AS pu_location_id,
    DOLocationID AS do_location_id,
    payment_type,
    fare_amount,
    tip_amount,
    tolls_amount,
    total_amount,
    filename AS source_file,
    CAST(regexp_extract(filename, '(\d{{4}})-(\d{{2}})', 1) AS INTEGER) AS source_year,
    CAST(regexp_extract(filename, '(\d{{4}})-(\d{{2}})', 2) AS INTEGER) AS source_month
FROM read_parquet('{green_glob}', filename = true, union_by_name = true);
""")

con.execute("""
CREATE OR REPLACE TEMP VIEW trips AS
SELECT * FROM yellow_trips
UNION ALL BY NAME
SELECT * FROM green_trips;
""")

print("Vistas creadas: yellow_trips, green_trips, trips")
print(con.execute("SELECT taxi_type, COUNT(*) AS n FROM trips GROUP BY 1 ORDER BY 1").fetchdf())


Vistas creadas: yellow_trips, green_trips, trips
  taxi_type         n
0     green    337114
1    yellow  29703355


## Exploración directa de archivos Parquet

### ¿Qué significa consultar directamente un archivo Parquet?

DuckDB puede ejecutar SQL sobre archivos Parquet con `read_parquet` **sin**
importarlos antes a una tabla persistente. El motor lee (de forma columnar) solo
lo necesario para responder la consulta.

### ¿Por qué puede ser útil con grandes volúmenes?

- lectura columnar: no hay que cargar todas las columnas;
- predicate pushdown cuando aplica (filtros cerca del almacenamiento);
- se evita una etapa obligatoria de carga/ETL previa;
- menor duplicación de datos en disco;
- es fácil agregar nuevos archivos al glob sin reescribir una base.


### Inventario de archivos (desde el sistema de archivos)

Ya se generó la tabla de inventario arriba. A continuación, conteos de registros
directamente sobre Parquet.


### Conteo de registros

**Objetivo:** cuantificar el volumen total y por tipo/mes.

**Fuente:** Parquet yellow/green 2026 vía vista `trips`.

**Consulta:** se muestra en la siguiente celda.

**Resultado:** se genera al ejecutar.

**Interpretación/decisión:** confirmar que hay datos suficientes de ambos tipos
antes de análisis posteriores.


In [6]:
counts_by_type = con.execute("""
SELECT taxi_type, COUNT(*) AS n_rows
FROM trips
GROUP BY taxi_type
ORDER BY taxi_type;
""").fetchdf()

counts_by_month = con.execute("""
SELECT taxi_type, source_year, source_month, COUNT(*) AS n_rows
FROM trips
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3;
""").fetchdf()

total_rows = int(counts_by_type["n_rows"].sum())
print(f"Registros totales: {total_rows:,}")
display(counts_by_type)
display(counts_by_month)


Registros totales: 30,040,469


,taxi_type,n_rows
0,green,337114
1,yellow,29703355


,taxi_type,source_year,source_month,n_rows
0,green,2026,1,40272
1,green,2026,2,37373
2,green,2026,3,44208
3,green,2026,4,44238
4,green,2026,5,44921
5,green,2026,6,44163
6,green,2026,7,41252
7,green,2026,8,40687
8,yellow,2026,1,3724889
9,yellow,2026,2,3399866


### Columnas y tipos

**Objetivo:** documentar el esquema de yellow, green y de la vista normalizada,
y señalar diferencias entre tipos.

**Fuente:** archivos Parquet originales + vista `trips`.

**Consulta:** `DESCRIBE` sobre `read_parquet` y sobre `trips`.

**Resultado:** tablas de esquema al ejecutar.

**Interpretación/decisión:** la vista normalizada unifica nombres (`tpep_*` /
`lpep_*` → `pickup_datetime` / `dropoff_datetime`) sin reescribir los archivos.


In [7]:
schema_yellow = con.execute(f"""
DESCRIBE SELECT * FROM read_parquet('{yellow_glob}', filename = true, union_by_name = true);
""").fetchdf()

schema_green = con.execute(f"""
DESCRIBE SELECT * FROM read_parquet('{green_glob}', filename = true, union_by_name = true);
""").fetchdf()

schema_trips = con.execute("DESCRIBE trips;").fetchdf()

print("Esquema yellow")
display(schema_yellow)
print("Esquema green")
display(schema_green)
print("Esquema vista normalizada (trips)")
display(schema_trips)

yellow_cols = set(schema_yellow["column_name"])
green_cols = set(schema_green["column_name"])
only_yellow = sorted(yellow_cols - green_cols)
only_green = sorted(green_cols - yellow_cols)
print("Solo en yellow:", only_yellow)
print("Solo en green :", only_green)


Esquema yellow


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


Esquema green


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,lpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,lpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,store_and_fwd_flag,VARCHAR,YES,None,None,None
4,RatecodeID,BIGINT,YES,None,None,None
5,PULocationID,INTEGER,YES,None,None,None
6,DOLocationID,INTEGER,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None
8,trip_distance,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


Esquema vista normalizada (trips)


,column_name,column_type,null,key,default,extra
0,taxi_type,VARCHAR,YES,None,None,None
1,pickup_datetime,TIMESTAMP,YES,None,None,None
2,dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,pu_location_id,INTEGER,YES,None,None,None
6,do_location_id,INTEGER,YES,None,None,None
7,payment_type,BIGINT,YES,None,None,None
8,fare_amount,DOUBLE,YES,None,None,None
9,tip_amount,DOUBLE,YES,None,None,None


Solo en yellow: ['Airport_fee', 'tpep_dropoff_datetime', 'tpep_pickup_datetime']
Solo en green : ['ehail_fee', 'lpep_dropoff_datetime', 'lpep_pickup_datetime', 'trip_type']


### Muestra de registros

**Objetivo:** inspeccionar unas pocas filas de cada tipo.

**Fuente:** vista `trips`.

**Consulta:** `LIMIT 5` por tipo.

**Resultado:** muestra pequeña al ejecutar.

**Interpretación/decisión:** verificar que la normalización de columnas y
`source_file` / mes se vean coherentes.


In [8]:
sample_df = con.execute("""
(
    SELECT * FROM trips WHERE taxi_type = 'yellow' LIMIT 5
)
UNION ALL BY NAME
(
    SELECT * FROM trips WHERE taxi_type = 'green' LIMIT 5
);
""").fetchdf()
display(sample_df)


,taxi_type,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,pu_location_id,do_location_id,payment_type,fare_amount,tip_amount,tolls_amount,total_amount,source_file,source_year,source_month
0,yellow,2026-01-01 00:54:04,2026-01-01 00:59:37,1,0.97,239,238,1,7.2,3.66,0.0,15.86,/workspace/data/raw/yellow/2026/yellow_tripdat...,2026,1
1,yellow,2026-01-01 00:34:04,2026-01-01 00:39:47,0,0.90,163,162,2,7.9,0.00,0.0,13.65,/workspace/data/raw/yellow/2026/yellow_tripdat...,2026,1
2,yellow,2026-01-01 00:57:06,2026-01-01 01:05:59,0,1.40,43,237,1,10.7,2.50,0.0,18.95,/workspace/data/raw/yellow/2026/yellow_tripdat...,2026,1
3,yellow,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,142,209,1,38.7,11.11,0.0,55.56,/workspace/data/raw/yellow/2026/yellow_tripdat...,2026,1
4,yellow,2026-01-01 00:27:13,2026-01-01 00:40:43,0,2.16,88,144,1,13.5,3.85,0.0,23.10,/workspace/data/raw/yellow/2026/yellow_tripdat...,2026,1
5,green,2026-01-01 00:27:58,2026-01-01 00:55:16,2,6.20,65,233,1,31.7,7.50,0.0,45.20,/workspace/data/raw/green/2026/green_tripdata_...,2026,1
6,green,2026-01-01 00:44:33,2026-01-01 01:32:56,5,5.36,66,188,1,50.0,10.20,0.0,61.20,/workspace/data/raw/green/2026/green_tripdata_...,2026,1
7,green,2026-01-01 00:23:45,2026-01-01 00:45:03,4,10.60,65,179,1,41.5,2.00,0.0,46.00,/workspace/data/raw/green/2026/green_tripdata_...,2026,1
8,green,2026-01-01 00:44:33,2026-01-01 01:00:45,1,4.20,42,141,2,19.8,0.00,0.0,25.05,/workspace/data/raw/green/2026/green_tripdata_...,2026,1
9,green,2026-01-01 00:46:04,2026-01-01 01:04:40,1,2.76,95,82,2,19.1,0.00,0.0,21.60,/workspace/data/raw/green/2026/green_tripdata_...,2026,1


## Calidad de datos inicial

No se eliminan registros todavía: el objetivo es **cuantificar** problemas.

Umbral de velocidad sospechosa: **100 mph**. Es una regla analítica de
detección (viajes con duración positiva y distancia no negativa). No implica
por sí sola que el registro sea definitivamente inválido.


### Datetimes nulos y orden temporal

**Objetivo:** medir nulos en pickup/dropoff y casos con dropoff ≤ pickup.

**Fuente:** vista `trips`.

**Consulta:** agregaciones con `FILTER`.

**Resultado:** al ejecutar.

**Interpretación/decisión:** estos casos suelen excluirse en análisis de
duración/velocidad.


In [9]:
datetime_quality = con.execute("""
SELECT
    taxi_type,
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE pickup_datetime IS NULL) AS null_pickup,
    COUNT(*) FILTER (WHERE dropoff_datetime IS NULL) AS null_dropoff,
    COUNT(*) FILTER (
        WHERE pickup_datetime IS NOT NULL
          AND dropoff_datetime IS NOT NULL
          AND dropoff_datetime <= pickup_datetime
    ) AS dropoff_le_pickup
FROM trips
GROUP BY taxi_type
ORDER BY taxi_type;
""").fetchdf()
display(datetime_quality)


,taxi_type,n_rows,null_pickup,null_dropoff,dropoff_le_pickup
0,green,337114,0,0,234
1,yellow,29703355,0,0,371683


### Distancia, tarifas y pasajeros

**Objetivo:** detectar distancias negativas/cero, montos negativos y pasajeros
negativos.

**Fuente:** vista `trips`.

**Consulta:** conteos con filtros.

**Resultado:** al ejecutar.

**Interpretación/decisión:** cero distancia puede ser válido en algunos casos
(errores de GPS o viajes cancelados); negativos suelen ser anomalías.


In [10]:
amount_quality = con.execute("""
SELECT
    taxi_type,
    COUNT(*) FILTER (WHERE trip_distance < 0) AS negative_distance,
    COUNT(*) FILTER (WHERE trip_distance = 0) AS zero_distance,
    COUNT(*) FILTER (WHERE fare_amount < 0) AS negative_fare,
    COUNT(*) FILTER (WHERE total_amount < 0) AS negative_total,
    COUNT(*) FILTER (WHERE passenger_count < 0) AS negative_passengers,
    COUNT(*) FILTER (WHERE passenger_count IS NULL) AS null_passengers,
    COUNT(*) FILTER (WHERE trip_distance IS NULL) AS null_distance,
    COUNT(*) FILTER (WHERE pu_location_id IS NULL) AS null_pu,
    COUNT(*) FILTER (WHERE do_location_id IS NULL) AS null_do,
    COUNT(*) FILTER (WHERE fare_amount IS NULL) AS null_fare,
    COUNT(*) FILTER (WHERE total_amount IS NULL) AS null_total
FROM trips
GROUP BY taxi_type
ORDER BY taxi_type;
""").fetchdf()
display(amount_quality)


,taxi_type,negative_distance,zero_distance,negative_fare,negative_total,negative_passengers,null_passengers,null_distance,null_pu,null_do,null_fare,null_total
0,green,0,12212,999,1023,0,48775,0,0,0,0,0
1,yellow,0,952231,157364,161835,0,7716688,0,0,0,0,0


### Duración excesiva y velocidad implícita

**Objetivo:** contar viajes con duración > 24 h y velocidad implícita > 100 mph
(solo con duración positiva y distancia ≥ 0).

**Fuente:** vista `trips`.

**Consulta:** derivadas de `epoch(dropoff) - epoch(pickup)`.

**Resultado:** al ejecutar.

**Interpretación/decisión:** el umbral de 100 mph es una regla de detección,
no una prueba definitiva de invalidez.


In [11]:
SPEED_THRESHOLD_MPH = 100

duration_speed = con.execute(f"""
SELECT
    taxi_type,
    COUNT(*) FILTER (
        WHERE pickup_datetime IS NOT NULL
          AND dropoff_datetime IS NOT NULL
          AND epoch(dropoff_datetime) - epoch(pickup_datetime) > 24 * 3600
    ) AS duration_over_24h,
    COUNT(*) FILTER (
        WHERE pickup_datetime IS NOT NULL
          AND dropoff_datetime IS NOT NULL
          AND epoch(dropoff_datetime) - epoch(pickup_datetime) > 0
          AND trip_distance >= 0
          AND trip_distance
              / ((epoch(dropoff_datetime) - epoch(pickup_datetime)) / 3600.0)
              > {SPEED_THRESHOLD_MPH}
    ) AS speed_over_{SPEED_THRESHOLD_MPH}_mph
FROM trips
GROUP BY taxi_type
ORDER BY taxi_type;
""").fetchdf()
display(duration_speed)
print(f"Umbral de velocidad usado: {SPEED_THRESHOLD_MPH} mph")


,taxi_type,duration_over_24h,speed_over_100_mph
0,green,4,1023
1,yellow,263,6501


Umbral de velocidad usado: 100 mph


### Percentiles extremos

**Objetivo:** ubicar colas de distancia, tarifa y monto total.

**Fuente:** vista `trips`.

**Consulta:** `approx_quantile`.

**Resultado:** al ejecutar.

**Interpretación/decisión:** percentiles altos ayudan a definir cortes futuros
sin basarse solo en máximos absolutos.


In [12]:
percentiles = con.execute("""
SELECT
    taxi_type,
    approx_quantile(trip_distance, 0.01) AS dist_p01,
    approx_quantile(trip_distance, 0.50) AS dist_p50,
    approx_quantile(trip_distance, 0.99) AS dist_p99,
    approx_quantile(fare_amount, 0.01) AS fare_p01,
    approx_quantile(fare_amount, 0.50) AS fare_p50,
    approx_quantile(fare_amount, 0.99) AS fare_p99,
    approx_quantile(total_amount, 0.01) AS total_p01,
    approx_quantile(total_amount, 0.50) AS total_p50,
    approx_quantile(total_amount, 0.99) AS total_p99
FROM trips
GROUP BY taxi_type
ORDER BY taxi_type;
""").fetchdf()
display(percentiles.round(2))


,taxi_type,dist_p01,dist_p50,dist_p99,fare_p01,fare_p50,fare_p99,total_p01,total_p50,total_p99
0,green,0.0,2.07,17.81,0.00,13.22,80.87,4.91,20.46,97.66
1,yellow,0.0,1.86,19.53,3.16,15.80,81.53,7.73,23.61,105.87


## Conclusiones de la exploración

Cifras obtenidas de la ejecución real sobre Parquet 2026 (ene–ago):

1. **Ambiente:** Docker Compose levanta `analysis` (JupyterLab + DuckDB) en
   `:8888` y `metabase` en `:3000`. Versiones detectadas: Python 3.11.17,
   DuckDB 1.1.3, Pandas 2.2.3, PyArrow 18.1.0.
2. **Descarga:** 16 archivos publicados (8 yellow + 8 green), ~495.66 MB.
   Meses sep–dic aún no publicados. Verificación: **COMPLETO** respecto a lo
   disponible en la TLC. Re-ejecutar el script omite los archivos válidos.
3. **Volumen:** 30,040,469 registros (yellow 29,703,355; green 337,114),
   consultados en memoria con vistas normalizadas, sin base persistente.
4. **Esquema:** yellow aporta `tpep_*` y `Airport_fee`; green aporta `lpep_*`,
   `ehail_fee` y `trip_type`. La vista `trips` unifica datetimes y montos.
5. **Calidad (sin borrar aún):**
   - datetimes nulos: 0 en ambos tipos;
   - dropoff ≤ pickup: ~371.7k yellow, 234 green;
   - distancia 0: ~952k yellow, ~12.2k green (sin distancias negativas);
   - tarifa/total negativos: del orden de 157k–162k en yellow y ~1k en green;
   - `passenger_count` nulo: ~7.7M yellow, ~49k green;
   - duración > 24 h: 263 yellow, 4 green;
   - velocidad implícita > 100 mph (regla analítica): 6,501 yellow, 1,023 green.
6. Medianas razonables (p50 distancia ~1.9–2.1 mi; p50 total ~20–24 USD), con
   colas largas en p99.

Consultas principales también en `sql/exploration.sql`.
